# 01 - C-MAPSS exploration

NASA turbofan degradation, the training data for **Model A** (remaining useful life).

Four subsets differing in operating conditions and fault modes. Everything below
imports from `twinmind.data`; no preprocessing logic is redefined here.


In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from twinmind.utils.config import load_config
from twinmind.utils.paths import figures_dir
from twinmind.utils.seed import set_seed

set_seed(42)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
FIGURES = figures_dir()
DPI = 150

def save(fig, name):
    """Save a figure to reports/figures as a 150-dpi PNG."""
    path = FIGURES / f"{name}.png"
    fig.savefig(path, dpi=DPI, bbox_inches="tight")
    print(f"saved {path.relative_to(path.parents[2])}")
    return path

config = load_config("data")


In [ ]:
from twinmind.data.loaders import load_cmapss
from twinmind.data.windowing import (
    compute_rul, compute_test_rul, drop_constant_sensors,
    fit_operating_condition_normaliser, make_windows, split_by_unit,
)

cfg = config.cmapss
SENSORS, SETTINGS = cfg.sensor_columns, cfg.setting_columns

train = {s: load_cmapss(s, "train") for s in cfg.subsets}
test = {s: load_cmapss(s, "test") for s in cfg.subsets}

summary = pd.DataFrame(
    [
        {
            "subset": s,
            "train_rows": len(train[s]),
            "train_units": train[s]["unit"].nunique(),
            "test_rows": len(test[s]),
            "test_units": test[s]["unit"].nunique(),
            "min_len": int(train[s].groupby("unit").size().min()),
            "median_len": int(train[s].groupby("unit").size().median()),
            "max_len": int(train[s].groupby("unit").size().max()),
        }
        for s in cfg.subsets
    ]
)
summary

## Unit-length distribution per subset

How long each engine runs before failure.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
for ax, s in zip(axes.ravel(), cfg.subsets):
    lengths = train[s].groupby("unit").size()
    ax.hist(lengths, bins=30, color="#4C72B0", edgecolor="white")
    ax.axvline(lengths.median(), color="#C44E52", ls="--",
               label=f"median {int(lengths.median())}")
    ax.axvline(cfg.window_length, color="#55A868", ls=":",
               label=f"window {cfg.window_length}")
    ax.set_title(f"{s} - {train[s]['unit'].nunique()} units")
    ax.set_xlabel("trajectory length (cycles)")
    ax.legend(fontsize=8)
fig.suptitle("C-MAPSS unit-length distribution by subset", fontweight="bold")
fig.tight_layout()
save(fig, "01_cmapss_unit_length_histogram");

## Per-sensor variance and which sensors get dropped

Constant sensors are **detected** from the training split, never hard-coded: the
flat set differs by subset, because FD002/FD004 vary across six operating regimes.


In [ ]:
dropped_by_subset = {}
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, s in zip(axes.ravel(), cfg.subsets):
    kept, dropped = drop_constant_sensors(
        train[s], SENSORS, cfg.constant_variance_threshold
    )
    dropped_by_subset[s] = dropped
    var = train[s][SENSORS].var()
    colours = ["#C44E52" if c in dropped else "#4C72B0" for c in SENSORS]
    ax.bar(range(len(SENSORS)), var.reindex(SENSORS).clip(lower=1e-30), color=colours)
    ax.axhline(cfg.constant_variance_threshold, color="black", ls="--",
               label=f"threshold {cfg.constant_variance_threshold:g}")
    ax.set_yscale("log")
    ax.set_xticks(range(len(SENSORS)))
    ax.set_xticklabels([c.split("_")[1] for c in SENSORS], fontsize=7)
    ax.set_title(f"{s} - dropping {len(dropped)}: "
                 f"{sorted(int(c.split('_')[1]) for c in dropped)}")
    ax.set_xlabel("sensor"); ax.set_ylabel("variance (log)")
    ax.legend(fontsize=8)
fig.suptitle("Per-sensor training variance; red bars are dropped", fontweight="bold")
fig.tight_layout()
save(fig, "01_cmapss_sensor_variance");

pd.Series({s: sorted(int(c.split("_")[1]) for c in d)
           for s, d in dropped_by_subset.items()}).to_frame("dropped sensors")

## The piecewise-linear RUL cap

Before the cap, the target claims a brand-new engine is "300 cycles from failure",
which the sensors cannot support. Capping at 125 asks the model only for what the
data can actually tell it.


In [ ]:
sub = cfg.primary_subset
raw_rul = (
    train[sub].groupby("unit")["cycle"].transform("max") - train[sub]["cycle"]
)
capped = compute_rul(train[sub], cap=cfg.rul_cap)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(raw_rul, bins=60, color="#4C72B0", edgecolor="white")
axes[0].set_title(f"{sub}: RUL before the cap")
axes[0].set_xlabel("RUL (cycles)"); axes[0].set_ylabel("rows")
axes[0].axvline(cfg.rul_cap, color="#C44E52", ls="--", label=f"cap {cfg.rul_cap}")
axes[0].legend()

axes[1].hist(capped, bins=60, color="#55A868", edgecolor="white")
axes[1].set_title(f"{sub}: RUL after the piecewise-linear cap")
axes[1].set_xlabel("RUL (cycles)")
pct = 100 * (capped == cfg.rul_cap).mean()
axes[1].annotate(f"{pct:.1f}% of rows pinned at {cfg.rul_cap}",
                 xy=(0.45, 0.85), xycoords="axes fraction",
                 bbox=dict(boxstyle="round", fc="white", ec="#C44E52"))
fig.suptitle("Piecewise-linear RUL target", fontweight="bold")
fig.tight_layout()
save(fig, "01_cmapss_rul_distribution_cap");

## Sensor trajectories for five sample units, coloured by RUL

In [ ]:
kept, _ = drop_constant_sensors(train[sub], SENSORS, cfg.constant_variance_threshold)
show = kept[:6]
frame = train[sub].copy()
frame["rul"] = compute_rul(frame, cap=cfg.rul_cap)
sample_units = sorted(frame["unit"].unique())[:5]

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, sensor in zip(axes.ravel(), show):
    for unit in sample_units:
        block = frame[frame["unit"] == unit]
        sc = ax.scatter(block["cycle"], block[sensor], c=block["rul"],
                        cmap="viridis_r", s=4)
    ax.set_title(sensor); ax.set_xlabel("cycle")
fig.colorbar(sc, ax=axes, label="RUL (cycles)", shrink=0.8)
fig.suptitle(f"{sub}: sensor trajectories for 5 units, coloured by RUL",
             fontweight="bold")
save(fig, "01_cmapss_sensor_trajectories");

## Regime structure on FD002

FD002 and FD004 are flown across six operating conditions. The op settings cluster
cleanly, which is what `fit_operating_condition_normaliser` exploits.


In [ ]:
multi = cfg.multi_regime_subsets[0]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(train[multi][SETTINGS[0]], train[multi][SETTINGS[1]], s=2, alpha=0.3)
axes[0].set_xlabel(SETTINGS[0]); axes[0].set_ylabel(SETTINGS[1])
axes[0].set_title(f"{multi}: operating settings form discrete regimes")

norm = fit_operating_condition_normaliser(
    train[multi], SENSORS, SETTINGS, n_conditions=cfg.n_operating_conditions
)
before = train[multi]["sensor_2"]
after = norm.transform(train[multi])["sensor_2"]
axes[1].hist(before, bins=60, alpha=0.6, label="raw", color="#4C72B0")
ax2 = axes[1].twiny()
ax2.hist(after, bins=60, alpha=0.6, label="within-regime z-scored", color="#55A868")
axes[1].set_title("sensor_2 before (blue) and after (green) regime normalisation")
fig.suptitle("Operating-condition structure", fontweight="bold")
fig.tight_layout()
save(fig, "01_cmapss_operating_regimes");

## End-to-end preprocessing check

The exact path `train_health.py` will take.

In [ ]:
frame = train[sub].copy()
frame["rul"] = compute_rul(frame, cap=cfg.rul_cap)
tr, va = split_by_unit(frame, cfg.val_fraction, seed=config.splits.seed)
assert not set(tr["unit"]) & set(va["unit"])

kept, dropped = drop_constant_sensors(tr, SENSORS, cfg.constant_variance_threshold)
norm = fit_operating_condition_normaliser(tr, kept, SETTINGS, n_conditions=1)
X, y, units = make_windows(norm.transform(tr), kept, window=cfg.window_length)
Xv, yv, _ = make_windows(norm.transform(va), kept, window=cfg.window_length)

print(f"kept {len(kept)} sensors, dropped {len(dropped)}")
print(f"train windows {X.shape}  targets {y.shape}  units {len(np.unique(units))}")
print(f"val   windows {Xv.shape}  targets {yv.shape}")
print(f"target range [{y.min()}, {y.max()}]")